# CTF Challenge-Solving Assistant — Baseline LLM Agent

**Project:** A CTF challenge-solving assistant for beginner/intermediate text-based challenges  
**Authors:** Dahana Moz Ruiz, Maria Santos  

This notebook implements the **Task 2 Baseline LLM Agent** using:
- GPT-4o via OpenAI API (core reasoning)
- Chain-of-thought prompting
- Tool use: Base64 decode, Hex decode, Caesar brute-force, Python execution
- Structured output for evaluation

Datasets Used:
- **InterCode-CTF** (`ic_ctf.json`) --> incorporates 100 text-based picoCTF challenges, no Docker needed
- **NYU Dataset**

> **Dataset note:**

- InterCode-CTF's tasks were manually collected from picoCTF, so `ic_ctf.json` already covers the PicoCTF Writeups dataset from the proposal. No separate scraping needed. [InterCode-CTF & picoCTF from proposal had the same data]
- CTF GitHub from proposal was not used since docker enviroment can't be used on colab


## 1. Install Dependencies

In [ ]:
!pip install openai --quiet

**NYU CTF Bench metadata (challenge names, descriptions, flags, categories) without Docker**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Load NYU CTF Bench (no Docker required for metadata) ──────────────────────
!pip install nyuctf openai pandas numpy tqdm -q
print("Libraries installed.")

import os, shutil, json
from pathlib import Path
from nyuctf.dataset import CTFDataset
from nyuctf.challenge import CTFChallenge

LOCAL_NYUCTF_PATH  = Path.home() / '.nyuctf'
DRIVE_DATASET_PATH = '/content/drive/MyDrive/CTF_Dataset'

# Only copy JSON files — skips Docker images, binaries, and symlinks
# that caused the 10+ minute hang
print("Copying challenge metadata from Drive (JSON files only)...")
for json_file in Path(DRIVE_DATASET_PATH).rglob('*.json'):
    relative = json_file.relative_to(DRIVE_DATASET_PATH)
    dest = LOCAL_NYUCTF_PATH / relative
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(json_file), str(dest))

print("Done copying. Loading dataset...")

ds_dev  = CTFDataset(split="development")
ds_test = CTFDataset(split="test")

print(f"Dataset loaded!")
print(f"  Development split: {len(ds_dev.dataset)} challenges")
print(f"  Test split:        {len(ds_test.dataset)} challenges")

In [ ]:
# ── Convert NYU CTF Bench to eval-compatible format ───────────────────────────
NYU_TEXT_CATEGORIES = {"cry", "misc"}   # ← updated to match actual naming

nyu_eval_subset = []
for challenge_id in ds_dev.dataset.keys():
    chal = CTFChallenge(ds_dev.dataset.get(challenge_id), ds_dev.basedir)
    category = chal.category.lower()

    if category not in NYU_TEXT_CATEGORIES:
        continue

    nyu_eval_subset.append({
        "task_id":  challenge_id,
        "query":    chal.description,
        "gold":     chal.flag,
        "tags":     [chal.category],
        "source":   f"NYU CTF Bench / {challenge_id}",
    })

print(f"NYU text-solvable challenges: {len(nyu_eval_subset)}")
print(f"(from {len(ds_dev.dataset)} total in development split)")

## 2. API Key Setup

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("API key loaded from Colab Secrets")

## 3. Tool Implementations

These are the tools the agent can call to solve challenges. Each tool maps to a real CTF technique.

In [ ]:
import base64
import json
import re
import contextlib
from io import StringIO

# ── Tool 1: Base64 Decode ──────────────────────────────────────────────────────
def decode_base64(data: str) -> str:
    """Decode a base64-encoded string to plaintext."""
    try:
        padded = data.strip() + "=" * (-len(data.strip()) % 4)
        return base64.b64decode(padded).decode("utf-8", errors="replace")
    except Exception as e:
        return f"Error decoding base64: {e}"

# ── Tool 2: Hex Decode ─────────────────────────────────────────────────────────
def decode_hex(data: str) -> str:
    """Decode a hexadecimal string to plaintext."""
    try:
        clean = data.strip().replace(" ", "").replace("0x", "")
        return bytes.fromhex(clean).decode("utf-8", errors="replace")
    except Exception as e:
        return f"Error decoding hex: {e}"

# ── Tool 3: Caesar Cipher Brute-Force ─────────────────────────────────────────
def caesar_brute_force(ciphertext: str) -> str:
    """Try all 26 Caesar shifts and return all candidates."""
    results = []
    for shift in range(26):
        decoded = ""
        for ch in ciphertext:
            if ch.isalpha():
                base = ord('A') if ch.isupper() else ord('a')
                decoded += chr((ord(ch) - base - shift) % 26 + base)
            else:
                decoded += ch
        results.append(f"Shift {shift:2d}: {decoded}")
    return "\n".join(results)

# ── Tool 4: Python Code Execution (sandboxed) ──────────────────────────────────
def execute_python(code: str) -> str:
    """Execute Python code in a restricted environment and capture stdout."""
    blocked = ["os.system", "subprocess", "shutil", "__import__"]
    for b in blocked:
        if b in code:
            return f"Blocked: '{b}' is not allowed in sandbox."
    stdout_capture = StringIO()
    try:
        with contextlib.redirect_stdout(stdout_capture):
            exec(code, {"__builtins__": __builtins__})
        return stdout_capture.getvalue() or "(no output)"
    except Exception as e:
        return f"Execution error: {e}"

# ── Tool map for dispatching ───────────────────────────────────────────────────
TOOL_MAP = {
    "decode_base64": decode_base64,
    "decode_hex": decode_hex,
    "caesar_brute_force": caesar_brute_force,
    "execute_python": execute_python,
}

# Quick sanity checks
assert "picoCTF" in decode_base64("cGljb0NURntlbmNvZGluZ19pc19ub3RfZW5jcnlwdGlvbn0=")
assert "picoCTF" in decode_hex("706963 6f4354 467b68 33785f 6d3467 69637d")
print("✅ All tools loaded and verified")

## 4. System Prompt & Agent Loop

The agent uses **chain-of-thought prompting** and OpenAI's **function calling** to reason through challenges step by step, pick the right tool, and explain the solution.

In [ ]:
import openai

client = openai.OpenAI()
MODEL = "gpt-4o"

# ── OpenAI Function Schemas ────────────────────────────────────────────────────
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "decode_base64",
            "description": "Decode a base64-encoded string to plaintext.",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {"type": "string", "description": "The base64 string to decode."}
                },
                "required": ["data"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "decode_hex",
            "description": "Decode a hexadecimal string to plaintext.",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {"type": "string", "description": "The hex string to decode."}
                },
                "required": ["data"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "caesar_brute_force",
            "description": "Brute-force all 26 Caesar cipher shifts on a ciphertext.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ciphertext": {"type": "string", "description": "The Caesar-ciphered text."}
                },
                "required": ["ciphertext"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "execute_python",
            "description": "Execute a Python code snippet and return stdout. Use for XOR, custom math, or scripting.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {"type": "string", "description": "Python code to execute."}
                },
                "required": ["code"]
            }
        }
    }
]

# ── picoCTF System Prompt (InterCode-CTF / ic_ctf.json dataset) ────────────────
PICO_SYSTEM_PROMPT = """You are an expert CTF (Capture The Flag) teaching assistant helping students solve picoCTF challenges.

Your role is to:
1. IDENTIFY the challenge category (crypto, encoding, forensics, misc, web, etc.)
2. REASON step by step about how to approach it (chain-of-thought)
3. USE available tools when needed (base64 decode, hex decode, caesar brute-force, python execution)
4. EXPLAIN every step clearly so the student learns, not just gets the answer
5. OUTPUT the final flag if found

Always prioritize teaching over just solving. Explain WHY each approach works.

FLAG FORMAT RULES — follow these exactly:
- picoCTF challenges always use the format picoCTF{...} — always wrap your answer in picoCTF{} not flag{}
- The flag value may contain underscores, numbers, or special characters — output them exactly as-is
- Never surround the flag with backticks, quotes, or any other characters
- The flag on the ## FLAG line must be plain text only, e.g. picoCTF{some_value_here}

Format your final response as:
## Category: <type>
## Reasoning: <step-by-step thinking>
## Solution: <what you did and why>
## FLAG: picoCTF{...} or Not found — here's what to try next: ...
"""

# ── CSAW / NYU CTF System Prompt (NYU CTF Bench dataset) ──────────────────────
NYU_SYSTEM_PROMPT = """You are an expert CTF (Capture The Flag) teaching assistant helping students solve CSAW CTF challenges.

Your role is to:
1. IDENTIFY the challenge category (crypto, encoding, forensics, misc, web, etc.)
2. REASON step by step about how to approach it (chain-of-thought)
3. USE available tools when needed (base64 decode, hex decode, caesar brute-force, python execution)
4. EXPLAIN every step clearly so the student learns, not just gets the answer
5. OUTPUT the final flag if found

Always prioritize teaching over just solving. Explain WHY each approach works.

FLAG FORMAT RULES — follow these exactly:
- CSAW CTF flags always use the format flag{...} — always wrap your answer in flag{} not picoCTF{}
- The flag value may contain regex patterns, special characters, or punctuation — output them exactly as-is
- If the challenge involves regex, the answer IS likely a regex string wrapped in flag{} — do not treat it as a search pattern to apply
- Never surround the flag with backticks, quotes, or any other characters
- The flag on the ## FLAG line must be plain text only, e.g. flag{some_value_here}

Format your final response as:
## Category: <type>
## Reasoning: <step-by-step thinking>
## Solution: <what you did and why>
## FLAG: flag{...} or Not found — here's what to try next: ...
"""

# ── Default prompt alias (picoCTF, since ic_ctf is the primary dataset) ────────
SYSTEM_PROMPT = PICO_SYSTEM_PROMPT

# ── Agent Loop ─────────────────────────────────────────────────────────────────
def run_ctf_agent(challenge: str, verbose: bool = True, system_prompt: str = None) -> dict:
    """
    Run the baseline CTF agent on a challenge description.

    Args:
        challenge: The CTF challenge text.
        verbose: Whether to print intermediate steps.
        system_prompt: Override the default system prompt. If None, uses SYSTEM_PROMPT.

    Returns:
        dict with keys: category, reasoning, solution, flag, tool_calls, full_response
    """
    messages = [
        {"role": "system", "content": system_prompt or SYSTEM_PROMPT},
        {"role": "user", "content": f"Solve this CTF challenge:\n\n{challenge}"}
    ]

    tool_calls_log = []
    max_iterations = 6

    for iteration in range(max_iterations):
        if verbose:
            print(f"\n[Iteration {iteration + 1}] Calling {MODEL}...")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0.2,
        )

        msg = response.choices[0].message
        messages.append(msg)

        # No tool calls → final answer
        if not msg.tool_calls:
            final_text = msg.content or ""
            if verbose:
                print("\n" + "="*50)
                print("AGENT RESPONSE")
                print("="*50)
                print(final_text)
            return _parse_response(final_text, tool_calls_log)

        # Process each tool call
        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            arg_preview = list(fn_args.values())[0][:80] if fn_args else ""

            if verbose:
                print(f"  → Tool: {fn_name}")
                print(f"    Input: {arg_preview}")

            fn = TOOL_MAP.get(fn_name)
            result = fn(**fn_args) if fn else f"Unknown tool: {fn_name}"
            tool_calls_log.append({"tool": fn_name, "args": fn_args, "result": result})

            if verbose:
                print(f"    Output: {result[:200]}")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result
            })

    return {"error": "Max iterations reached", "tool_calls": tool_calls_log}


def _sanitize_flag(raw: str, dataset: str = "InterCode-CTF") -> str:
    """Strip backticks/quotes. Auto-wrap bare values using the correct
    flag format for the dataset (picoCTF{} for ic_ctf, flag{} for NYU)."""
    flag = raw.strip().strip('`').strip("'").strip('"').strip()
    # If it already has any {...} wrapper, leave it alone
    if re.search(r'\{.*\}', flag):
        return flag
    # If it's a not-found message, leave it alone
    if 'not found' in flag.lower():
        return flag
    # Bare value with no wrapper — add correct wrapper based on dataset
    if flag:
        if dataset == "NYU CTF Bench":
            flag = f"flag{{{flag}}}"
        else:
            flag = f"picoCTF{{{flag}}}"
    return flag


def _parse_response(text: str, tool_calls: list, dataset: str = "InterCode-CTF") -> dict:
    """Parse the agent's final markdown response into structured fields."""
    def extract(label):
        pattern = rf"##\s*{label}:\s*(.*?)(?=\n##|\Z)"
        m = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        return m.group(1).strip() if m else ""

    flag_match = re.search(r"FLAG:\s*(.+)", text, re.IGNORECASE)
    raw_flag   = flag_match.group(1).strip() if flag_match else "Not found"
    return {
        "category":      extract("Category"),
        "reasoning":     extract("Reasoning"),
        "solution":      extract("Solution"),
        "flag":          _sanitize_flag(raw_flag, dataset=dataset),
        "full_response": text,
        "tool_calls":    tool_calls,
    }

print("Agent loop ready")
print(f"  SYSTEM_PROMPT  → PICO_SYSTEM_PROMPT (picoCTF{{}} format, for ic_ctf.json)")
print(f"  NYU_SYSTEM_PROMPT → flag{{}} format, for NYU CTF Bench")


## 5. Run on Sample Challenges

Test the agent on four beginner-level challenge types.

In [ ]:
# ── Challenge 1: Base64 Encoding ───────────────────────────────────────────────
challenge_1 = """
I found this string online and my friend says it hides a flag. Can you help me figure out what it says?

cGljb0NURntlbmNvZGluZ19pc19ub3RfZW5jcnlwdGlvbn0=
"""

result_1 = run_ctf_agent(challenge_1)
print("\n🚩 FLAG:", result_1["flag"])

In [ ]:
# ── Challenge 2: Hex Decode ────────────────────────────────────────────────────
challenge_2 = """
My friend sent me this weird string and said there's a secret message inside. What does it say?

706963 6f4354 467b68 33785f 6d3467 69637d
"""

result_2 = run_ctf_agent(challenge_2)
print("\n🚩 FLAG:", result_2["flag"])

In [ ]:
# ── Challenge 3: Caesar Cipher ─────────────────────────────────────────────────
challenge_3 = """
This message was intercepted from the enemy. It looks like a simple substitution:

Cvkeg gxgtaqpg, vjg hncm ku: rkeoEVH{ecguct_ku_gcua}
"""

result_3 = run_ctf_agent(challenge_3)
print("\n🚩 FLAG:", result_3["flag"])

In [ ]:
# ── Challenge 4: XOR with Python ──────────────────────────────────────────────
challenge_4 = """
The flag was XOR'd byte-by-byte with the key 42. Here are the resulting byte values:

[90, 75, 90, 27, 14, 26, 27, 21, 26, 95, 95, 14, 95, 89, 90, 95, 95, 25, 15]

Can you recover the original flag?
"""

result_4 = run_ctf_agent(challenge_4)
print("\n🚩 FLAG:", result_4["flag"])

## 6. Load InterCode-CTF Dataset (`ic_ctf.json`)

This fetches the real dataset directly from the Princeton NLP GitHub repo — no Docker, no install needed, works in Colab.

**What's in it:** 100 text-based CTF challenges sourced from picoCTF, with fields:
- `query` — the challenge description (what you send to the agent)
- `gold` — the correct flag
- `tags` — category list (e.g. `["General Skills"]`, `["Cryptography"]`)
- `source` — URL to the original picoCTF problem
- `task_id` — unique identifier

> Since these came from picoCTF, this also replaces the separate PicoCTF Writeups dataset from the proposal.

In [ ]:
import requests
import json
from collections import defaultdict

# ── Fetch ic_ctf.json directly from GitHub ─────────────────────────────────────
IC_CTF_URL = "https://raw.githubusercontent.com/princeton-nlp/intercode/master/data/ctf/ic_ctf.json"

response = requests.get(IC_CTF_URL)
response.raise_for_status()
ic_ctf_raw = response.json()

print(f"✅ Loaded {len(ic_ctf_raw)} challenges from ic_ctf.json")
print(f"   Fields per item: {list(ic_ctf_raw[0].keys())}")
print()

# ── Normalize into a clean list ────────────────────────────────────────────────
# ic_ctf.json is a dict keyed by task_id, not a list
if isinstance(ic_ctf_raw, dict):
    ic_ctf = [
        {
            "task_id":  task_id,
            "query":    item.get("query", ""),
            "gold":     item.get("gold", ""),
            "tags":     item.get("tags", []),
            "source":   item.get("source", ""),
        }
        for task_id, item in ic_ctf_raw.items()
    ]
else:
    ic_ctf = ic_ctf_raw

# ── Show category breakdown ────────────────────────────────────────────────────
category_counts = defaultdict(int)
for item in ic_ctf:
    for tag in item["tags"]:
        category_counts[tag] += 1

print("Category breakdown:")
for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    bar = "█" * count
    print(f"  {cat:<22} {bar} ({count})")

print()
print("Sample challenge:")
sample = ic_ctf[0]
print(f"  task_id : {sample['task_id']}")
print(f"  tags    : {sample['tags']}")
print(f"  gold    : {sample['gold']}")
print(f"  query   : {sample['query'][:200]}...")
print(f"  source  : {sample['source']}")

## 7. Batch Evaluation on InterCode-CTF Data & NYU CTF Data

Run the agent on challenges both datasets and compute all metrics from the proposal:
- **Task Success Rate** — % where exact flag is found
- **Partial Credit Score** — rubric-based score for reasoning + category + flag
- **Solve Rate by Category** — breakdown across General Skills, Crypto, Forensics, etc.

⚠️ Using `MAX_CHALLENGES` to control how many problems to run.

In [ ]:
import time

# ── Config ─────────────────────────────────────────────────────────────────────
MAX_CHALLENGES = 20        # ← set to len(ic_ctf) to run all 100
FILTER_CATEGORY = None     # ← e.g. "Cryptography" to run one category only, or None for all
DELAY_BETWEEN_CALLS = 1.0  # seconds between API calls (avoid rate limits)

# ── Text-solvable filter ───────────────────────────────────────────────────────
# Forensics, Reverse Engineering, Binary Exploitation, and most Web Exploitation
# challenges require actual binary/image files or a live server connection that
# a text-only baseline cannot access. Filtering to text-solvable categories gives
# a fairer picture of the baseline's true reasoning ability.
#
# Set TEXT_ONLY = False to run on all categories (matches original full eval).
TEXT_ONLY = True

TEXT_SOLVABLE_CATEGORIES = {"Cryptography", "General Skills"}

# ── Filter & slice dataset ─────────────────────────────────────────────────────
if TEXT_ONLY:
    subset = [c for c in ic_ctf if any(t in TEXT_SOLVABLE_CATEGORIES for t in c["tags"])]
    print(f"Text-only filter ON: {len(subset)} challenges (Cryptography + General Skills)")
elif FILTER_CATEGORY:
    subset = [c for c in ic_ctf if FILTER_CATEGORY in c["tags"]]
    print(f"Filtered to category '{FILTER_CATEGORY}': {len(subset)} challenges")
else:
    subset = ic_ctf
    print(f"Running on all {len(subset)} challenges (includes file-dependent ones)")

subset = subset[:MAX_CHALLENGES]
print(f"Running evaluation on {len(subset)} challenges...\n")

ic_ctf_subset = subset

# ── Partial Credit Rubric ──────────────────────────────────────────────────────
# Maps ic_ctf tags to the categories the agent identifies
TAG_TO_CATEGORY = {
    "General Skills":      "misc",
    "Cryptography":        "crypto",
    "Forensics":           "forensics",
    "Reverse Engineering": "rev",
    "Binary Exploitation": "pwn",
    "Web Exploitation":    "web",
}

def get_expected_category(tags: list) -> str:
    """Map ic_ctf tags to the normalized category the agent should identify."""
    for tag in tags:
        if tag in TAG_TO_CATEGORY:
            return TAG_TO_CATEGORY[tag].lower()
    return "misc"

def partial_credit(agent_result: dict, expected_category: str, gold_flag: str) -> float:
    """
    Score 0.0–1.0 based on the proposal rubric:
      0.4 pts — correct category identified
      0.3 pts — non-trivial reasoning produced (>50 chars)
      0.3 pts — exact flag match
    """
    score = 0.0
    predicted_cat = agent_result.get("category", "").lower()
    # Allow partial category matches (e.g. agent says "cryptography", expected "crypto")
    if expected_category in predicted_cat or predicted_cat in expected_category:
        score += 0.4
    if len(agent_result.get("reasoning", "")) > 50:
        score += 0.3
    predicted_flag = agent_result.get("flag", "").strip().lower()
    if predicted_flag == gold_flag.strip().lower():
        score += 0.3
    return round(score, 2)

# ── Run Evaluation Loop ────────────────────────────────────────────────────────
results_log = []

for dataset_name, eval_subset in [("InterCode-CTF", ic_ctf_subset), ("NYU CTF Bench", nyu_eval_subset)]:
    print(f"\n{'='*55}")
    print(f"  Running: {dataset_name} ({len(eval_subset)} challenges)")
    print(f"{'='*55}\n")

    for i, item in enumerate(eval_subset):
        expected_cat = get_expected_category(item["tags"])
        tag_label    = item["tags"][0] if item["tags"] else "unknown"
        print(f"[{i+1:>3}/{len(eval_subset)}] {tag_label:<22} | task: {item['task_id']}")

        try:
            agent_result = run_ctf_agent(item["query"], verbose=False)
            score  = partial_credit(agent_result, expected_cat, item["gold"])
            exact  = agent_result.get("flag", "").strip().lower() == item["gold"].strip().lower()
            status = "✅" if exact else "❌"
            print(f"         {status} Predicted: {agent_result.get('flag', 'Not found'):<35} | Gold: {item['gold']}  | Score: {score}")

            results_log.append({
                "dataset":        dataset_name,        # ← only new field
                "task_id":        item["task_id"],
                "category":       tag_label,
                "predicted_flag": agent_result.get("flag", "Not found"),
                "gold_flag":      item["gold"],
                "exact_match":    exact,
                "partial_score":  score,
                "reasoning_len":  len(agent_result.get("reasoning", "")),
                "source":         item["source"],
            })
        except Exception as e:
            print(f"         ⚠️  Error: {e}")
            results_log.append({
                "dataset":        dataset_name,        # ← only new field
                "task_id": item["task_id"], "category": tag_label,
                "predicted_flag": "ERROR", "gold_flag": item["gold"],
                "exact_match": False, "partial_score": 0.0,
                "reasoning_len": 0, "source": item["source"],
            })

    time.sleep(DELAY_BETWEEN_CALLS)

print(f"\n✅ Evaluation complete — {len(results_log)} challenges processed.")

In [ ]:
# ── Compute & Display All Metrics ──────────────────────────────────────────────
from collections import defaultdict

for dataset_name in ["InterCode-CTF", "NYU CTF Bench"]:
    ds_results = [r for r in results_log if r["dataset"] == dataset_name]
    if not ds_results:
        print(f"\n(no results for {dataset_name})\n")
        continue

    total         = len(ds_results)
    exact_matches = sum(r["exact_match"] for r in ds_results)
    task_success  = exact_matches / total
    avg_partial   = sum(r["partial_score"] for r in ds_results) / total

    by_cat = defaultdict(lambda: {"total": 0, "correct": 0, "partial_sum": 0.0})
    for r in ds_results:
        cat = r["category"]
        by_cat[cat]["total"]       += 1
        by_cat[cat]["partial_sum"] += r["partial_score"]
        if r["exact_match"]:
            by_cat[cat]["correct"] += 1

    print("=" * 55)
    print(f"  EVALUATION RESULTS — {dataset_name}")
    print("=" * 55)
    print(f"Challenges run  : {total}")
    print(f"Task Success    : {task_success:.1%}  ({exact_matches}/{total} exact flag matches)")
    print(f"Avg Partial     : {avg_partial:.2f} / 1.00")
    print()
    print("Solve Rate by Category:")
    print(f"  {'Category':<24} {'Solved':>6}  {'Total':>5}  {'Rate':>5}  {'Avg Partial':>11}")
    print("  " + "-" * 55)
    for cat, counts in sorted(by_cat.items(), key=lambda x: -x[1]["correct"]):
        rate  = counts["correct"] / counts["total"]
        avg_p = counts["partial_sum"] / counts["total"]
        bar   = "█" * counts["correct"] + "░" * (counts["total"] - counts["correct"])
        print(f"  {cat:<24} {counts['correct']:>6}  {counts['total']:>5}  {rate:>4.0%}   {avg_p:>6.2f}   {bar}")
    print("=" * 55)

    print("\nDetailed Results:")
    print(f"  {'#':<4} {'Task ID':<25} {'Cat':<22} {'OK':<3} {'Score':<6} {'Predicted Flag'}")
    print("  " + "-" * 90)
    for i, r in enumerate(ds_results, 1):
        mark      = "✅" if r["exact_match"] else "❌"
        predicted = r["predicted_flag"][:35]
        print(f"  {i:<4} {r['task_id']:<25} {r['category']:<22} {mark:<3} {r['partial_score']:<6} {predicted}")
    print()

**Benchmark**

We evaluate on InterCode-CTF, a standardized benchmark of 100 text-based picoCTF challenges. Due to API cost constraints we evaluate on a 20-challenge subset (text-solvable categories only: Cryptography and General Skills). Our zero-shot baseline achieves 25% task success rate and an average partial score of 0.64/1.00 on this subset. On the NYU CTF Bench development split (6 misc/crypto challenges), the agent achieves 0% task success rate with an average partial score of 0.42/1.00, reflecting the difficulty of context-free misc challenges that require cultural or environmental knowledge beyond the model's text-only reasoning.

In [ ]:
# ── Save results to CSV ────────────────────────────────────────────────────────
import csv

csv_path = "results.csv"
fieldnames = ["dataset", "task_id", "category", "predicted_flag", "gold_flag",
              "exact_match", "partial_score", "reasoning_len", "source"]

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results_log)

print(f"Results saved to {csv_path}")

**Evaluations**

We ran 3 evaluation test:
- Consistency Evaluation
- Category Identification Accuracy
- Reasoning Quality Score

## 9. Consistency Evaluation

Run the same challenges 3 times and check whether the agent produces the same flag and category each time. This tests **reliability** — an agent that gives different answers on the same input is not dependable, even if it is sometimes correct.

We use the NYU CTF Bench here since it is a standardized benchmark. Results are compared across runs using exact string match.

> ⚠️ This will make `3 × N` API calls. Keep `CONSISTENCY_SAMPLE` small (5–10) to manage cost.

In [ ]:
import time
from collections import Counter

# ── Config ─────────────────────────────────────────────────────────────────────
CONSISTENCY_RUNS   = 3     # number of times to run each challenge
CONSISTENCY_DELAY  = 1.0   # seconds between calls

# Use only challenges the agent was able to solve in the main eval
# This gives a meaningful consistency test — we want to know if correct
# answers are reliable, not just whether wrong answers are consistently wrong.
solved_tasks = {str(r["task_id"]) for r in results_log if r["exact_match"]}
sample = [item for item in ic_ctf_subset if str(item["task_id"]) in solved_tasks]
print(f"Solved challenges available for consistency test: {len(sample)}")
print(f"Task IDs: {[item['task_id'] for item in sample]}\n")

consistency_log = []
print(f"Running {CONSISTENCY_RUNS} runs x {len(sample)} challenges = {CONSISTENCY_RUNS * len(sample)} API calls\n")

for item in sample:
    flags      = []
    categories = []

    for run in range(1, CONSISTENCY_RUNS + 1):
        result = run_ctf_agent(item["query"], verbose=False, system_prompt= SYSTEM_PROMPT)
        flags.append(result.get("flag", "Not found").strip().lower())
        categories.append(result.get("category", "").strip().lower())
        time.sleep(CONSISTENCY_DELAY)

    flag_consistent = len(set(flags)) == 1
    cat_consistent  = len(set(categories)) == 1
    flag_majority   = Counter(flags).most_common(1)[0][0]
    cat_majority    = Counter(categories).most_common(1)[0][0]
    gold            = item["gold"].strip().lower()

    consistency_log.append({
        "task_id":          item["task_id"],
        "gold_flag":        item["gold"],
        "flags_per_run":    flags,
        "cats_per_run":     categories,
        "flag_consistent":  flag_consistent,
        "cat_consistent":   cat_consistent,
        "majority_flag":    flag_majority,
        "majority_cat":     cat_majority,
        "majority_correct": flag_majority == gold,
        "all_correct":      all(f == gold for f in flags),
    })

    f_mark = "OK" if flag_consistent else "INCONSISTENT"
    c_mark = "OK" if cat_consistent  else "INCONSISTENT"
    all_ok = all(f == gold for f in flags)
    print(f"Task: {item['task_id']}  (gold: {item['gold']})")
    print(f"  Flag  [{f_mark}]: {flags}")
    print(f"  Cat   [{c_mark}]: {categories}")
    print(f"  All correct: {'YES' if all_ok else 'NO'}")
    print()

# ── Summary ────────────────────────────────────────────────────────────────────
n = len(consistency_log)
flag_cons_rate = sum(r["flag_consistent"] for r in consistency_log) / n
cat_cons_rate  = sum(r["cat_consistent"]  for r in consistency_log) / n
majority_acc   = sum(r["majority_correct"] for r in consistency_log) / n
all_correct    = sum(r["all_correct"]      for r in consistency_log) / n

print("=" * 50)
print("  CONSISTENCY RESULTS — solved challenges")
print("=" * 50)
print(f"Challenges tested      : {n} (previously solved)")
print(f"Flag consistency rate  : {flag_cons_rate:.0%}  (same flag all {CONSISTENCY_RUNS} runs)")
print(f"Category consistency   : {cat_cons_rate:.0%}  (same category all {CONSISTENCY_RUNS} runs)")
print(f"Always correct rate    : {all_correct:.0%}  (correct on all {CONSISTENCY_RUNS} runs)")
print(f"Majority-vote accuracy : {majority_acc:.0%}  (majority flag == gold)")
print("=" * 50)

## 10. Category Identification Accuracy

Evaluate how well the agent identifies the correct challenge category, **independent of whether it finds the flag**. This is scored separately because category identification is an important sub-skill — an agent that consistently misclassifies challenges will apply the wrong tools and strategies.

Scored across both datasets using the full `results_log` from the main evaluation.

In [ ]:
from collections import defaultdict

# ── Normalize category names across both datasets ──────────────────────────────
CATEGORY_NORM = {
    "general skills":      "misc",
    "cryptography":        "crypto",
    "forensics":           "forensics",
    "reverse engineering": "rev",
    "binary exploitation": "pwn",
    "web exploitation":    "web",
    "misc":   "misc",
    "crypto": "crypto",
    "cry":    "crypto",
}

def normalize_category(raw: str) -> str:
    raw = raw.strip().lower()
    for key, val in CATEGORY_NORM.items():
        if key in raw:
            return val
    return raw

# ── Score from results_log ─────────────────────────────────────────────────────
cat_results     = defaultdict(lambda: {"correct": 0, "total": 0})
overall_correct = 0

for r in results_log:
    gold_cat      = normalize_category(r["category"])
    predicted_cat = normalize_category(r.get("category", ""))
    is_correct    = (gold_cat == predicted_cat
                     or gold_cat in predicted_cat
                     or predicted_cat in gold_cat)

    cat_results[gold_cat]["total"]   += 1
    cat_results[gold_cat]["correct"] += int(is_correct)
    overall_correct += int(is_correct)

overall_acc = overall_correct / len(results_log) if results_log else 0

print("=" * 50)
print("  CATEGORY IDENTIFICATION ACCURACY")
print("=" * 50)
print(f"Overall accuracy : {overall_acc:.1%}  ({overall_correct}/{len(results_log)})")
print()
print(f"  {'Category':<14} {'Correct':>7}  {'Total':>5}  {'Accuracy':>8}")
print("  " + "-" * 40)
for cat, counts in sorted(cat_results.items(), key=lambda x: -x[1]["total"]):
    acc = counts["correct"] / counts["total"]
    bar = "X" * counts["correct"] + "." * (counts["total"] - counts["correct"])
    print(f"  {cat:<14} {counts['correct']:>7}  {counts['total']:>5}  {acc:>7.0%}   {bar}")
print("=" * 50)
print()
print("Note: category accuracy is scored independently of flag success.")
print("An agent can correctly identify the category but still fail to find the flag")

## 11. Reasoning Quality Score

Manually rate a sample of agent reasoning explanations on a **1–5 scale** across three dimensions:

| Dimension | What it measures |
|---|---|
| Relevance | Does the reasoning address the actual challenge? |
| Correctness | Are the technical steps and concepts accurate? |
| Clarity | Is the explanation clear enough for a beginner to follow? |

This is important because the project goal is **teaching**, not just solving. A high task success rate with poor explanations does not meet the project objective.

**How to use:**
1. Run the first cell below — it prints 10 reasoning samples
2. Read each one and fill in your scores in the `SCORES` dict in the second cell
3. Run the second cell to see the summary and save to CSV

In [ ]:
# ── Step 1: Print reasoning samples to review ─────────────────────────────────
# Pulls challenges with the longest reasoning from results_log for a fair sample.

REASONING_SAMPLE_SIZE = 10

candidates = sorted(
    [r for r in results_log if r.get("reasoning_len", 0) > 100],
    key=lambda x: -x["reasoning_len"]
)[:REASONING_SAMPLE_SIZE]

print(f"Reviewing {len(candidates)} reasoning samples")
print("Rate each: Relevance, Correctness, Clarity (1=poor, 3=ok, 5=excellent)")
print("=" * 60)

# Build a lookup of query text for re-running the agent
query_lookup = {str(item["task_id"]): item["query"]
                for item in (ic_ctf_subset + nyu_eval_subset)}

for i, r in enumerate(candidates, 1):
    print(f"\n[{i}] Task: {r['task_id']}  |  Dataset: {r['dataset']}  |  Category: {r['category']}")
    print(f"    Gold flag  : {r['gold_flag']}")
    print(f"    Predicted  : {r['predicted_flag']}")
    print(f"    Exact match: {'YES' if r['exact_match'] else 'NO'}")
    print()

    query = query_lookup.get(str(r["task_id"]))
    if query:
        prompt = NYU_SYSTEM_PROMPT if r["dataset"] == "NYU CTF Bench" else SYSTEM_PROMPT
        result = run_ctf_agent(query, verbose=False, system_prompt=prompt)
        reasoning = result.get("reasoning", "(no reasoning found)")
        print(f"    REASONING:")
        print(f"    {reasoning[:800]}")
    else:
        print("    (query not found — re-run main eval first)")

    print()
    print("-" * 60)

In [ ]:
# ── Step 2: Fill in your scores after reading the samples above ───────────────
# Format: "task_id": {"relevance": 1-5, "correctness": 1-5, "clarity": 1-5}
#
# Scoring rubric:
#   1 = poor    (irrelevant, wrong, or confusing)
#   2 = weak    (partially relevant but mostly off)
#   3 = ok      (addresses the challenge but misses key steps)
#   4 = good    (mostly correct and clear, minor gaps)
#   5 = excellent (fully relevant, technically accurate, beginner-friendly)

SCORES = {
    # Task 12 — RSA small N. Correctly identifies the vulnerability and all steps
    # (factor N, compute phi, derive d, decrypt). Technically accurate, no values to work with.
    "12": {"relevance": 5, "correctness": 5, "clarity": 4},

    # Task 24 — codebook challenge. Correctly identifies code.py + codebook.txt structure
    # and execution approach. Generic but appropriate given no file access.
    "24": {"relevance": 5, "correctness": 4, "clarity": 4},

    # Task 29 — netcat hashing. Correctly identifies the connect → receive → hash → send flow.
    # Clear and accurate, just can't execute live.
    "29": {"relevance": 5, "correctness": 4, "clarity": 4},

    # Task 26 — Python syntax fix. No reasoning produced at all. Asks for the script
    # rather than attempting to reason about common syntax errors.
    "26": {"relevance": 1, "correctness": 1, "clarity": 1},

    # Task 11 — binary/strings challenge. Correctly identifies strings, file, hexdump, objdump.
    # Good beginner explanation of binary forensics approaches.
    "11": {"relevance": 5, "correctness": 5, "clarity": 4},

    # Task 9 — netcat challenge. Correctly identifies nc connection and encoding detection approach.
    # Clear step-by-step but can't connect to execute.
    "9": {"relevance": 5, "correctness": 4, "clarity": 4},

    # Task 2014f-msc-pps — NYU pun challenge. No reasoning produced at all.
    "2014f-msc-pps": {"relevance": 1, "correctness": 1, "clarity": 1},

    # Task 18 — hex to decimal conversion. Correctly identifies and explains the conversion,
    # successfully finds the flag. Clean and beginner-friendly.
    "18": {"relevance": 5, "correctness": 5, "clarity": 5},

    # Task 28 — netcat glitch challenge. Correctly identifies nc connection and interprets
    # "glitching" as obfuscated output. Good reasoning, blocked by live connection.
    "28": {"relevance": 4, "correctness": 4, "clarity": 4},

    # Task 6 — Python script + password. Correctly identifies the structure but has no
    # script or password to reason about specifically. Stays generic as a result.
    "6": {"relevance": 3, "correctness": 3, "clarity": 3},
}

# ── Step 3: Run this cell after filling SCORES to see the summary ─────────────
if not SCORES:
    print("Fill in the SCORES dict above with your ratings, then re-run this cell.")
else:
    relevance   = [v["relevance"]   for v in SCORES.values()]
    correctness = [v["correctness"] for v in SCORES.values()]
    clarity     = [v["clarity"]     for v in SCORES.values()]
    overall     = [(r + c + cl) / 3 for r, c, cl in zip(relevance, correctness, clarity)]

    print("=" * 45)
    print("  REASONING QUALITY SCORES (1-5 scale)")
    print("=" * 45)
    print(f"Challenges rated : {len(SCORES)}")
    print(f"Avg Relevance    : {sum(relevance)   / len(relevance):.2f} / 5")
    print(f"Avg Correctness  : {sum(correctness) / len(correctness):.2f} / 5")
    print(f"Avg Clarity      : {sum(clarity)     / len(clarity):.2f} / 5")
    print(f"Avg Overall      : {sum(overall)     / len(overall):.2f} / 5")
    print("=" * 45)
    print()
    print(f"  {'Task ID':<28} {'Rel':>4} {'Cor':>4} {'Cla':>4} {'Avg':>5}")
    print("  " + "-" * 48)
    for task_id, s in SCORES.items():
        avg = (s['relevance'] + s['correctness'] + s['clarity']) / 3
        print(f"  {str(task_id):<28} {s['relevance']:>4} {s['correctness']:>4} {s['clarity']:>4} {avg:>5.2f}")

    import csv
    with open("reasoning_quality_scores.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["task_id", "relevance", "correctness", "clarity", "avg"])
        writer.writeheader()
        for task_id, s in SCORES.items():
            avg = (s['relevance'] + s['correctness'] + s['clarity']) / 3
            writer.writerow({"task_id": task_id, **s, "avg": round(avg, 2)})
    print("\nScores saved to reasoning_quality_scores.csv")

**Discussion of system limitations**

*Execution limitations*

- Can't run files, scripts, or binaries — challenges requiring you to execute code.py, fix and run a script, or use strings/objdump on a binary all fail
- Can't make network connections — any netcat (nc) challenge is unsolvable regardless of how good the reasoning is

*Reasoning limitations*

- Misses cultural/contextual clues — the coinslot/Obama challenge shows the model defaults to technical decoding when the answer is a pop culture reference
- Hallucinates approaches — when no encoded data is present, the model invents a decoding problem rather than saying it lacks context
- Flag format confusion — without explicit per-dataset prompting, it mixes up picoCTF{} and flag{} wrappers, tanking exact match scores even on correctly-solved challenges

*Dataset limitations*

- ic_ctf.json is text-only by design, so the ~25% solve rate is the ceiling for a text-only agent — it's not representative of real CTF performance where most challenges require files or live infrastructure
- NYU CTF Bench misc/crypto categories have vague or context-free descriptions that give the agent nothing to reason about


## 12. Test Your Own Challenge

Paste any CTF challenge below and run the cell to see how the agent handles it.

In [ ]:
# ── Paste your challenge here ──────────────────────────────────────────────────
my_challenge = """
Paste your CTF challenge description here...
"""

result = run_ctf_agent(my_challenge, verbose=True)
print("\n" + "="*50)
print("🚩 FLAG:", result["flag"])
print("Category:", result["category"])

# **Task 3 Advanced Agent Design**

Part 1: Lightweight Fine-Tuning

Students perform lightweight model adaptation using methods such as:
- LoRA
- adapter tuning
- **instruction tuning**
- **prompt tuning**

Since GPT-4o is a closed model, weight-based methods (LoRA, adapters) are not accessible.

**Prompt Tuning**

In [ ]:
FEW_SHOT_EXAMPLES = [
    {
        "challenge": "I found this string online and my friend says it hides a flag.\ncGljb0NURntlbmNvZGluZ19pc19ub3RfZW5jcnlwdGlvbn0=",
        "response": """## Category: encoding
        ## Reasoning:
        1. The string ends with '=' which is a padding character used in Base64 encoding.
        2. Base64 encodes binary data as ASCII text using A-Z, a-z, 0-9, +, /.
        3. I will decode it using the base64 tool.
        ## Solution: Decoded the Base64 string using the decode_base64 tool.
        ## FLAG: picoCTF{encoding_is_not_encryption}"""
            },
            {
                "challenge": "My friend sent me this weird string. What does it say?\n706963 6f4354 467b68 33785f 6d3467 69637d",
                "response": """## Category: encoding
        ## Reasoning:
        1. The string consists of groups of hex digits separated by spaces.
        2. Each pair of hex digits represents one ASCII byte.
        3. I will strip spaces and decode using the hex tool.
        ## Solution: Decoded the hex string using the decode_hex tool.
        ## FLAG: picoCTF{h3x_m4gic}"""
    },
    {
        "challenge": "This message was intercepted. Decode it:\nCvkeg gxgtaqpg, vjg hncm ku: rkeoEVH{ecguct_ku_gcua}",
        "response": """## Category: crypto
        ## Reasoning:
        1. The text looks like English with shifted letters — classic Caesar cipher pattern.
        2. I will brute-force all 26 shifts using the caesar_brute_force tool.
        3. Shift 2 produces readable English: 'picoCTF{caesar_is_easy}'.
        ## Solution: Identified Caesar cipher, brute-forced shift of 2.
        ## FLAG: picoCTF{caesar_is_easy}"""
    },
]

def build_few_shot_prompt(base_prompt: str, examples: list) -> str:
    """Inject solved examples into the system prompt for few-shot tuning."""
    shots = "\n\n".join(
        f"EXAMPLE {i+1}:\nChallenge: {ex['challenge']}\n\nExpected response:\n{ex['response']}"
        for i, ex in enumerate(examples)
    )
    return base_prompt + f"\n\n--- SOLVED EXAMPLES (follow this reasoning style) ---\n\n{shots}\n\n--- END EXAMPLES ---"

PICO_FEW_SHOT_PROMPT = build_few_shot_prompt(PICO_SYSTEM_PROMPT, FEW_SHOT_EXAMPLES)
NYU_FEW_SHOT_PROMPT  = build_few_shot_prompt(NYU_SYSTEM_PROMPT,  FEW_SHOT_EXAMPLES)

print(f"Zero-shot prompt length : {len(PICO_SYSTEM_PROMPT)} chars")
print(f"Few-shot prompt length  : {len(PICO_FEW_SHOT_PROMPT)} chars")
print(f"Examples injected       : {len(FEW_SHOT_EXAMPLES)}")

In [ ]:
# comparing few-shot vs zero-shot on identical challenges
fewshot_results_log = []

for dataset_name, eval_subset in [("InterCode-CTF", ic_ctf_subset), ("NYU CTF Bench", nyu_eval_subset)]:
    print(f"\n{'='*55}")
    print(f"  Few-Shot Eval: {dataset_name} ({len(eval_subset)} challenges)")
    print(f"{'='*55}\n")

    for i, item in enumerate(eval_subset):
        sys_prompt = NYU_FEW_SHOT_PROMPT if dataset_name == "NYU CTF Bench" else PICO_FEW_SHOT_PROMPT
        tag_label  = item["tags"][0] if item["tags"] else "unknown"
        print(f"[{i+1:>3}/{len(eval_subset)}] {tag_label:<22} | task: {item['task_id']}")

        try:
            agent_result = run_ctf_agent(item["query"], verbose=False, system_prompt=sys_prompt)
            pred_flag = agent_result.get("flag", "").strip().lower()
            gold_flag = item["gold"].strip().lower()
            exact     = pred_flag == gold_flag
            score     = partial_credit(agent_result, get_expected_category(item["tags"]), item["gold"])
            status    = "✅" if exact else "❌"
            print(f"         {status} Predicted: {agent_result.get('flag', 'Not found'):<35} | Gold: {item['gold']}  | Score: {score}")

            fewshot_results_log.append({
                "dataset":        dataset_name,
                "task_id":        item["task_id"],
                "category":       tag_label,
                "predicted_flag": agent_result.get("flag", "Not found"),
                "gold_flag":      item["gold"],
                "exact_match":    exact,
                "partial_score":  score,
                "reasoning_len":  len(agent_result.get("reasoning", "")),
                "source":         item["source"],
            })
        except Exception as e:
            print(f"         ⚠️  Error: {e}")
            fewshot_results_log.append({
                "dataset": dataset_name, "task_id": item["task_id"], "category": tag_label,
                "predicted_flag": "ERROR", "gold_flag": item["gold"],
                "exact_match": False, "partial_score": 0.0, "reasoning_len": 0, "source": item["source"],
            })
        time.sleep(DELAY_BETWEEN_CALLS)

print(f"\n✅ Few-shot evaluation complete — {len(fewshot_results_log)} challenges processed.")

In [ ]:
print("=" * 60)
print("  PROMPT TUNING COMPARISON: Zero-Shot vs Few-Shot")
print("=" * 60)

for dataset_name in ["InterCode-CTF", "NYU CTF Bench"]:
    zs = [r for r in results_log         if r["dataset"] == dataset_name]
    fs = [r for r in fewshot_results_log if r["dataset"] == dataset_name]
    if not zs or not fs:
        continue

    zs_acc     = sum(r["exact_match"]   for r in zs) / len(zs)
    fs_acc     = sum(r["exact_match"]   for r in fs) / len(fs)
    zs_partial = sum(r["partial_score"] for r in zs) / len(zs)
    fs_partial = sum(r["partial_score"] for r in fs) / len(fs)

    print(f"\n  {dataset_name}")
    print(f"  {'Metric':<25} {'Zero-Shot':>10} {'Few-Shot':>10} {'Delta':>8}")
    print(f"  {'-'*55}")
    print(f"  {'Task Success Rate':<25} {zs_acc:>9.1%} {fs_acc:>9.1%} {fs_acc - zs_acc:>+7.1%}")
    print(f"  {'Avg Partial Score':<25} {zs_partial:>10.2f} {fs_partial:>10.2f} {fs_partial - zs_partial:>+8.2f}")

print("\n" + "=" * 60)

**Dataset used for tuning**
We used the InterCode-CTF dataset (ic_ctf.json) for both evaluation and in-context tuning. Three representative solved examples were selected from the text-solvable categories (base64 decoding, hex decoding, and Caesar cipher) and injected directly into the system prompt as demonstrations. No separate training split was created — this is in-context adaptation rather than weight-based fine-tuning.

**Training setup**
No model weights were modified. Prompt tuning was implemented by prepending 3 challenge-response demonstration pairs to the base system prompt, increasing prompt length from 1,165 to 2,819 characters. The model (GPT-4o), temperature (0.2), max iterations (6), and all other parameters remained identical to the baseline to ensure a controlled comparison.

**Improvement compared to baseline**
Few-shot prompt tuning did not improve exact match rate — task success remained at 25% on InterCode-CTF and 0% on NYU CTF Bench. However, average partial score improved by +0.08 on NYU CTF Bench (0.42 → 0.50), indicating that the demonstrations improved the structure and quality of the agent's reasoning on harder, more ambiguous challenges even when the flag could not be retrieved. The unchanged exact match rate confirms that the agent's primary bottleneck is the inability to execute files or connect to live services which are constraints that prompting strategies alone cannot overcome.

**Part 2: Tool-Based Agent**

Students must build an agent that integrates external tools.
- We will be integrating retrieval systems (RAG)

We extend the baseline by adding a retrieval tool backed by a FAISS vector index of solved CTF writeups sourced from the CTF GitHub dataset. This addresses the issues of when it cannot solve a challenge directly. It retrieves a structurally similar solved example and use it as context.

The multi-step reasoning flow is:
1. Agent reads the challenge and attempts known tools (base64, hex, caesar, python exec)
2. If stuck, calls `retrieve_similar_challenge` to find the most similar past solved problem
3. Uses the retrieved writeup as context to refine its approach

This integrates the **CTF GitHub dataset** (the fourth dataset from our proposal).

In [ ]:
# ── Part 2: RAG Tool — build retrieval corpus from CTF GitHub + ic_ctf holdout ─
!pip install faiss-cpu sentence-transformers --quiet

In [ ]:
# Part 2: RAG Tool — install dependencies
import json
import numpy as np
import faiss
import requests
from sentence_transformers import SentenceTransformer

# Intended to use CTF GitHub dataset but it had no structured data that we could use
# Repurposed the 80 unused challenges from InterCode-CTF [ic_ctf holdout]

corpus_challenges = [
    item for item in ic_ctf
    if item["task_id"] not in {str(c["task_id"]) for c in ic_ctf_subset}
]

CTF_WRITEUPS = [
    {
        "title":       f"CTF Task {item['task_id']}",
        "category":    item["tags"][0] if item["tags"] else "misc",
        "description": item["query"],
        "solution":    f"The flag is {item['gold']}",
        "flag":        item["gold"],
    }
    for item in corpus_challenges
]

print(f"Corpus size      : {len(CTF_WRITEUPS)} challenges (ic_ctf holdout)")
print(f"Eval subset size : {len(ic_ctf_subset)} challenges (not in corpus)")
print(f"Total ic_ctf     : {len(ic_ctf)} challenges")

# ── Build FAISS index ──────────────────────────────────────────────────────────
print("\nLoading sentence encoder...")
encoder = SentenceTransformer("all-MiniLM-L6-v2")

corpus_texts = [
    f"{w['title']}. {w['description']} Solution: {w['solution']}"
    for w in CTF_WRITEUPS
]

print("Encoding corpus...")
corpus_embeddings = encoder.encode(corpus_texts, convert_to_numpy=True)
corpus_embeddings = corpus_embeddings / np.linalg.norm(
    corpus_embeddings, axis=1, keepdims=True
)

index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
index.add(corpus_embeddings)

print(f"\n✅ RAG index built — {len(CTF_WRITEUPS)} writeups indexed from ic_ctf holdout")

In [ ]:
# ── Tool 5: RAG Retrieval
def retrieve_similar_challenge(query: str, top_k: int = 2) -> str:
    """Retrieve the most similar solved CTF writeups from the corpus."""
    q_emb = encoder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    distances, indices = index.search(q_emb, top_k)

    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
        w = CTF_WRITEUPS[idx]
        results.append(
            f"[Similar Challenge {rank}] (similarity: {dist:.2f})\n"
            f"Title    : {w['title']}\n"
            f"Category : {w['category']}\n"
            f"Problem  : {w['description']}\n"
            f"Solution : {w['solution']}"
        )
    return "\n\n".join(results)

# ── Extended tool map with RAG
TOOL_MAP_RAG = {**TOOL_MAP, "retrieve_similar_challenge": retrieve_similar_challenge}

# ── Extended OpenAI function schema
TOOLS_RAG = TOOLS + [
    {
        "type": "function",
        "function": {
            "name": "retrieve_similar_challenge",
            "description": (
                "Search a database of solved CTF writeups for challenges similar to the current one. "
                "Use this when you are unsure how to approach the challenge or when standard tools "
                "have not produced a result. Returns the top matching solved examples with solutions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A description of the current challenge to search for similar solved examples."
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# RAG-enabled agent loop
def run_rag_agent(challenge: str, verbose: bool = True, system_prompt: str = None) -> dict:
    """
    RAG-extended CTF agent. Same as run_ctf_agent but with retrieve_similar_challenge
    available as an additional tool, enabling multi-step reasoning via retrieval.
    """
    messages = [
        {"role": "system", "content": system_prompt or SYSTEM_PROMPT},
        {"role": "user", "content": f"Solve this CTF challenge:\n\n{challenge}"}
    ]

    tool_calls_log = []
    max_iterations = 8  # slightly higher to allow retrieval + follow-up

    for iteration in range(max_iterations):
        if verbose:
            print(f"\n[Iteration {iteration + 1}] Calling {MODEL}...")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS_RAG,
            tool_choice={"type": "function", "function": {"name": "retrieve_similar_challenge"}} if iteration == 0 else "auto",
            temperature=0.2,
        )

        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            final_text = msg.content or ""
            if verbose:
                print("\n" + "="*50)
                print("AGENT RESPONSE")
                print("="*50)
                print(final_text)
            return _parse_response(final_text, tool_calls_log)

        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            arg_preview = list(fn_args.values())[0][:80] if fn_args else ""

            if verbose:
                print(f"  → Tool: {fn_name}")
                print(f"    Input: {arg_preview}")

            fn = TOOL_MAP_RAG.get(fn_name)
            result = fn(**fn_args) if fn else f"Unknown tool: {fn_name}"
            tool_calls_log.append({"tool": fn_name, "args": fn_args, "result": result})

            if verbose:
                print(f"    Output: {result[:300]}")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result
            })

    return {"error": "Max iterations reached", "tool_calls": tool_calls_log}

print("RAG agent ready")
print(f"   Tools available: {list(TOOL_MAP_RAG.keys())}")

In [ ]:
# ── Evaluate RAG agent on same subset
import time

rag_results_log = []

for dataset_name, eval_subset in [("InterCode-CTF", ic_ctf_subset), ("NYU CTF Bench", nyu_eval_subset)]:
    print(f"\n{'='*55}")
    print(f"  RAG Agent Eval: {dataset_name} ({len(eval_subset)} challenges)")
    print(f"{'='*55}\n")

    for i, item in enumerate(eval_subset):
        sys_prompt = NYU_SYSTEM_PROMPT if dataset_name == "NYU CTF Bench" else PICO_SYSTEM_PROMPT
        tag_label  = item["tags"][0] if item["tags"] else "unknown"
        print(f"[{i+1:>3}/{len(eval_subset)}] {tag_label:<22} | task: {item['task_id']}")

        try:
            agent_result = run_rag_agent(item["query"], verbose=False, system_prompt=sys_prompt)
            pred_flag = agent_result.get("flag", "").strip().lower()
            gold_flag = item["gold"].strip().lower()
            exact     = pred_flag == gold_flag
            score     = partial_credit(agent_result, get_expected_category(item["tags"]), item["gold"])
            status    = "✅" if exact else "❌"
            rag_used  = any(tc["tool"] == "retrieve_similar_challenge" for tc in agent_result.get("tool_calls", []))
            print(f"         {status} {'[RAG]' if rag_used else '     '} Predicted: {agent_result.get('flag', 'Not found'):<35} | Gold: {item['gold']}  | Score: {score}")

            rag_results_log.append({
                "dataset":        dataset_name,
                "task_id":        item["task_id"],
                "category":       tag_label,
                "predicted_flag": agent_result.get("flag", "Not found"),
                "gold_flag":      item["gold"],
                "exact_match":    exact,
                "partial_score":  score,
                "reasoning_len":  len(agent_result.get("reasoning", "")),
                "rag_used":       rag_used,
                "source":         item["source"],
            })
        except Exception as e:
            print(f"         ⚠️  Error: {e}")
            rag_results_log.append({
                "dataset": dataset_name, "task_id": item["task_id"], "category": tag_label,
                "predicted_flag": "ERROR", "gold_flag": item["gold"],
                "exact_match": False, "partial_score": 0.0,
                "reasoning_len": 0, "rag_used": False, "source": item["source"],
            })
        time.sleep(DELAY_BETWEEN_CALLS)

print(f"\nRAG evaluation complete — {len(rag_results_log)} challenges processed.")

In [ ]:
# ── Three-way comparison: Baseline vs Few-Shot vs RAG
print("=" * 70)
print("  FULL COMPARISON: Zero-Shot vs Few-Shot Prompt Tuning vs RAG Agent")
print("=" * 70)

for dataset_name in ["InterCode-CTF", "NYU CTF Bench"]:
    zs  = [r for r in results_log         if r["dataset"] == dataset_name]
    fs  = [r for r in fewshot_results_log if r["dataset"] == dataset_name]
    rag = [r for r in rag_results_log     if r["dataset"] == dataset_name]
    if not zs:
        continue

    def metrics(log):
        if not log:
            return 0, 0
        return (sum(r["exact_match"] for r in log) / len(log),
                sum(r["partial_score"] for r in log) / len(log))

    zs_acc,  zs_p  = metrics(zs)
    fs_acc,  fs_p  = metrics(fs)
    rag_acc, rag_p = metrics(rag)

    rag_used_count = sum(r.get("rag_used", False) for r in rag)

    print(f"\n  {dataset_name}")
    print(f"  {'Metric':<25} {'Zero-Shot':>10} {'Few-Shot':>10} {'RAG':>10}")
    print(f"  {'-'*60}")
    print(f"  {'Task Success Rate':<25} {zs_acc:>9.1%} {fs_acc:>9.1%} {rag_acc:>9.1%}")
    print(f"  {'Avg Partial Score':<25} {zs_p:>10.2f} {fs_p:>10.2f} {rag_p:>10.2f}")
    if rag:
        print(f"  {'RAG tool invocations':<25} {'—':>10} {'—':>10} {rag_used_count:>9}/{len(rag)}")

print("\n" + "=" * 70)

The RAG tool was invoked on all 26 challenges (20/20 InterCode-CTF, 6/6 NYU CTF Bench). Using the ic_ctf holdout set as the retrieval corpus, task success rate jumped from 25% to 75% on InterCode-CTF — a +50% improvement over the zero-shot baseline — while NYU CTF Bench remained at 0%. Avg partial score on InterCode-CTF stayed comparable to baseline (0.66 → 0.60), and NYU improved slightly (0.42 → 0.43).

The substantial gain on InterCode-CTF demonstrates that retrieval from a closely matched corpus (held-out challenges from the same dataset) provides highly relevant context that enables the agent to solve challenges it previously could not. NYU CTF Bench showed no exact match improvement, consistent with those challenges requiring cultural knowledge or live connectivity that no retrieval corpus can substitute for.

This finding confirms that retrieval-augmented approaches are most effective when the corpus is drawn from the same distribution as the test set. The ic_ctf holdout split provides a natural and well-matched retrieval source, and the results suggest RAG is the strongest of the three approaches evaluated for text-solvable CTF challenges.

**Task 4: Evaluation and Ablation Study**



In [ ]:
#Ablation study

ablation_results = []

for dataset_name, eval_subset in [("InterCode-CTF", ic_ctf_subset), ("NYU CTF Bench", nyu_eval_subset)]:
    sys_prompt = NYU_SYSTEM_PROMPT if dataset_name == "NYU CTF Bench" else PICO_SYSTEM_PROMPT
    for item in eval_subset:
        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": f"Solve this CTF challenge:\n\n{item['query']}"}
        ]
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0.2,
            )
            result_text = response.choices[0].message.content or ""
            parsed = _parse_response(result_text, [])
            pred = parsed.get("flag", "").strip().lower()
            gold = item["gold"].strip().lower()
            exact = pred == gold
            score = partial_credit(parsed, get_expected_category(item["tags"]), item["gold"])
            ablation_results.append({
                "dataset": dataset_name, "task_id": item["task_id"],
                "exact_match": exact, "partial_score": score,
            })
        except Exception as e:
            ablation_results.append({
                "dataset": dataset_name, "task_id": item["task_id"],
                "exact_match": False, "partial_score": 0.0,
            })
        time.sleep(DELAY_BETWEEN_CALLS)

In [ ]:
## Full Evaluation + Ablation Summary

print("=" * 65)
print("  TASK 4: EVALUATION AND ABLATION STUDY")
print("=" * 65)

# F1 = task success rate for exact-match binary classification
for dataset_name in ["InterCode-CTF", "NYU CTF Bench"]:
    zs  = [r for r in results_log         if r["dataset"] == dataset_name]
    fs  = [r for r in fewshot_results_log if r["dataset"] == dataset_name]
    rag = [r for r in rag_results_log     if r["dataset"] == dataset_name]
    abl = [r for r in ablation_results    if r["dataset"] == dataset_name]

    def summarize(log):
        if not log: return 0, 0, 0
        acc = sum(r["exact_match"] for r in log) / len(log)
        return round(acc, 3), round(acc, 3), round(acc, 3)  # accuracy, F1, TSR all equal for exact match

    print(f"\n  {dataset_name}")
    print(f"  {'Method':<30} {'Accuracy':>9} {'F1':>6} {'Task Success':>13} {'Avg Partial':>12}")
    print(f"  {'-'*72}")
    for label, log in [
        ("Baseline (Zero-Shot)",       zs),
        ("Fine-tuned (Few-Shot)",       fs),
        ("Tool-based Agent (RAG)",      rag),
        ("Ablation: No Tools",          abl),
    ]:
        if not log: continue
        acc = sum(r["exact_match"] for r in log) / len(log)
        avg_p = sum(r["partial_score"] for r in log) / len(log)
        print(f"  {label:<30} {acc:>8.1%} {acc:>6.1%} {acc:>12.1%} {avg_p:>12.2f}")

print("\n" + "=" * 65)
print("  ABLATION FINDINGS")
print("=" * 65)
print("  Removing retrieval  → RAG vs Zero-Shot shows retrieval contribution")
print("  Removing tools      → Zero-Shot vs No-Tools shows tool contribution")
print("  Removing CoT prompt → Few-Shot partial score drop shows prompt impact")